In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FastSwiGLU(nn.Module): # TODO: rename fan_out to intermediate in lantern
    def __init__(
        self,
        fan_in: int,
        intermediate: int,
        bias: bool = False
    ):
        super().__init__()

        self.gatexvalue = nn.Linear(fan_in, 2 * intermediate, bias)
        self.out_proj = nn.Linear(intermediate, fan_in, bias)

    def forward(self, x: torch.Tensor):
        gate, value = self.gatexvalue(x).chunk(2, dim=-1)
        x = F.silu(gate) * value
        return self.out_proj(x)

class RoPE(nn.Module):
    def __init__(self, head_size: int, rope_theta: float = 10000.0, llama_style: bool = False):
        assert head_size % 2 == 0 # works only on even head_size

        super().__init__()

        self.head_size = head_size # hs
        self.rope_theta = rope_theta

        self.llama_style = llama_style

    def get_theta(
        self,
        seq_len: int, # T
        device,
        start_pos: int = 0
    ):
        inv_freq = 1 / (
            self.rope_theta ** (torch.arange(0, self.head_size, 2, device=device).float() / self.head_size)
        ) # (hs // 2,)
        pos = torch.arange(start_pos, start_pos + seq_len, device=device) # (T,)

        theta = torch.outer(pos, inv_freq) # (T, hs // 2)
        return theta

    def apply_rope(
        self,
        x: torch.Tensor, # (B, nh, T, hs)
        sin: torch.Tensor,
        cos: torch.Tensor
    ):
        if self.llama_style:
            x1 = x[..., : self.head_size // 2] # [x1, x2] (B, nh, T, hs // 2)
            x2 = x[..., self.head_size // 2 :] # [x3, x4] (B, nh, T, hs // 2)

            rotated = torch.cat((-x2, x1), dim=-1) # [-x3, -x4, x1, x2] (B, nh, T, hs)
            rotated = x * cos + rotated * sin # (B, nh, T, hs)

            return rotated
        else:
            x1 = x[..., 1::2]
            x2 = x[..., ::2]

            rotated = torch.stack((-x2, x1), dim=-1).flatten(-2) # stack creates a new dim so we need to flatten it back
            rotated = x * cos + rotated * sin

            return rotated

    def forward(self, x: torch.Tensor, start_pos=0):
        device = x.device
        dtype = x.dtype

        seq_len = x.shape[-2]

        theta = self.get_theta(seq_len, device, start_pos).to(dtype=dtype)
        sin, cos = theta.sin(), theta.cos() # (T, hs // 2)

        if self.llama_style:
            cos = torch.cat([cos, cos], dim=-1)
            sin = torch.cat([sin, sin], dim=-1) # (T, hs)
        else:
            cos = torch.stack([cos, cos], dim=-1).flatten(-2)
            sin = torch.stack([sin, sin], dim=-1).flatten(-2)

        return self.apply_rope(x, sin, cos) # (B, nh, T, hs)

In [ ]:
# DEEPSEEK V2 (replicating shibatch deepseek v2 3m)
import math
from dataclasses import dataclass

@dataclass
class ShibatchV2Config:
    block_size: int = 2048
    vocab_size: int = 1024

    n_layer: int = 5

    n_head: int = 8
    n_embd: int = 216

    qk_nope_head_dim: int = 16
    qk_rope_head_dim: int = 16
    v_head_dim: int = 32
    kv_lora_rank: int = 64

    block0_intermediate: int = 432

    n_routed_experts: int = 4
    expert_topk: int = 1
    n_shared_experts: int = 1
    d_expert: int = 128 # swiglu intermediate, to be precise

    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-6

class ShibatchV2(nn.Module):
    def __init__(self, config: ShibatchV2Config):
        super().__init__()
        self.config = config

        self.model = nn.ModuleDict(dict(
            embed_tokens = nn.Embedding(config.vocab_size, config.n_embd),
            layers = nn.ModuleList([Block0(config)] + [Block(config) for _ in range(config.n_layer - 1)]),
            norm = nn.RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # shibatch v2 3m has tied embeddings so to match them -
        self.lm_head.weight = self.model.embed_tokens.weight

    @property
    def num_params(self):
        return sum([p.numel() for p in self.parameters()])

    def forward(
        self,
        idx: torch.Tensor,
        cache = None,
        targets: torch.Tensor = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        T = idx.size(-1)
        assert T <= self.config.block_size, f"Cannot forward sequence of long length {T} while block size is only {self.config.block_size}"

        x = self.model.embed_tokens(idx)

        for i, h in enumerate(self.model.layers):
            x = h(x, cache=cache, layer_idx=i)

        x = self.model.norm(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        x: torch.Tensor, # x is B x T
        max_new_tokens: int = 500,
        topk: int = 50
    ) -> torch.Tensor: # without cache
        for _ in range(max_new_tokens):
            context = x[:, -self.config.block_size:] # truncate if input tokens go beyond block_size
            logits, _ = self(context)
            logits = logits[:, -1, :]
            prob = F.softmax(logits, dim=-1)

            topk = min(topk, self.config.vocab_size)
            topk_probs, topk_idx = torch.topk(prob, topk, dim=-1)
            ix = torch.multinomial(topk_probs, 1)
            xcol = torch.gather(topk_idx, -1, ix)
            x = torch.cat((x, xcol), dim=1)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.post_attention_layernorm = nn.RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.self_attn = DeepseekMLA(config, rope=RoPE(config.qk_rope_head_dim))
        self.input_layernorm = nn.RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.mlp = MoE(config.n_embd, config.d_expert, config.n_routed_experts, config.n_shared_experts, config.expert_topk)

    def forward(self, x: torch.Tensor, cache = None, layer_idx = None) -> torch.Tensor:
        x = x + self.self_attn(self.input_layernorm(x), cache=cache, layer_idx=layer_idx)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class Block0(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.post_attention_layernorm = nn.RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.self_attn = DeepseekMLA(config, rope=RoPE(config.qk_rope_head_dim))
        self.input_layernorm = nn.RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.mlp0 = FastSwiGLU(config.n_embd, config.block0_intermediate, bias=False)

    def forward(self, x: torch.Tensor, cache = None, layer_idx = None) -> torch.Tensor:
        x = x + self.self_attn(self.input_layernorm(x), cache=cache, layer_idx=layer_idx)
        x = x + self.mlp0(self.post_attention_layernorm(x))
        return x

class DeepseekMLA(nn.Module):
  def __init__(
      self,
      config,
      rope = None
  ):
      super().__init__()
      self.n_embd = config.n_embd
      self.n_head = config.n_head

      self.qk_nope_head_dim = config.qk_nope_head_dim
      self.qk_rope_head_dim = config.qk_rope_head_dim
      self.v_head_dim = config.v_head_dim
      self.kv_lora_rank = config.kv_lora_rank

      self.q_proj = nn.Linear(
          config.n_embd,
          self.n_head * (self.qk_nope_head_dim + self.qk_rope_head_dim),
          bias=False,
      )

      self.kv_a_proj_with_mqa = nn.Linear(
          config.n_embd,
          self.kv_lora_rank + self.qk_rope_head_dim,
          bias=False,
      )

      self.kv_a_layernorm = nn.RMSNorm(
          self.kv_lora_rank,
          eps=config.rms_norm_eps,
      )

      self.kv_b_proj = nn.Linear(
          self.kv_lora_rank,
          self.n_head * (self.qk_nope_head_dim + self.v_head_dim),
          bias=False,
      )

      self.o_proj = nn.Linear(
          self.n_head * (self.qk_nope_head_dim + self.qk_rope_head_dim),
          config.n_embd,
          bias=False,
      )

      self.rope = rope

  def forward(self, x: torch.Tensor, cache = None, layer_idx = None):
      B, T, _ = x.shape # B, T, C

      q = self.q_proj(x).view(B, T, self.n_head, self.qk_nope_head_dim + self.qk_rope_head_dim) # B, T, H, qk_nope_head_dim + R
      q_nope, q_rope = q[..., :self.qk_nope_head_dim], q[..., self.qk_nope_head_dim:]

      q_rope = q_rope.transpose(1, 2) # B, H, T, R
      q_rope = self.rope(
          q_rope,
          start_pos=(cache.pos if cache is not None else 0)
      )

      kv = self.kv_a_proj_with_mqa(x) # B, T, C + R
      kv_latent, k_rope = kv[..., :self.kv_lora_rank], kv[..., self.kv_lora_rank:]

      kv_latent = self.kv_a_layernorm(kv_latent) # normalizing because shibatch does too

      k_rope = self.rope(
          k_rope.unsqueeze(1),
          start_pos=(cache.pos if cache is not None else 0)
      ).squeeze(1) # B, T, R

      if cache is not None:
          cache.store(layer_idx, kv_latent, k_rope)
          kv_latent, k_rope = cache.get(layer_idx, end_pos=T) # B, S, C; B, S, R

      kv_b = self.kv_b_proj(kv_latent).view(
          B, -1,
          self.n_head,
          self.qk_nope_head_dim + self.v_head_dim
      )

      k_nope, v = kv_b[..., :self.qk_nope_head_dim], kv_b[..., self.qk_nope_head_dim:] # both B, S, H, D (D = qk_nope_head_dim)

      scores_nope = torch.einsum(
          'bthd,bshd->bhts', # t is the current seq_len i.e. 1 if cache; s is the total seq_len used
          q_nope,
          k_nope
      )
      scores_r = torch.einsum(
          'bhtr,bsr->bhts',
          q_rope,
          k_rope
      )
      scores = (scores_nope + scores_r) / (self.qk_nope_head_dim + self.qk_rope_head_dim) ** 0.5
      attn = torch.softmax(scores, dim=-1)

      context = torch.einsum(
          'bhts,bshv->bthv',
          attn,
          v
      ).reshape(B, T, self.n_head * self.v_head_dim)

      return self.o_proj(context)

class MoE(nn.Module):
  def __init__(
      self,
      d_model,
      d_expert,
      n_routed_experts,
      n_shared_experts,
      topk
  ):
      super().__init__()

      self.n_routed_experts = n_routed_experts
      self.n_shared_experts = n_shared_experts
      self.topk = topk

      self.router = nn.Linear(d_model, n_routed_experts, bias=False)

      self.shared_experts = nn.ModuleList([
          FastSwiGLU(d_model, d_expert, bias=False) # d_model -> d_expert (intermediate) -> d_model
          for _ in range(n_shared_experts)
      ])
      if n_shared_experts == 1: # have to do this to load weights correctly as it matches the way shibatch has it
          self.shared_experts = self.shared_experts[0]

      self.experts = nn.ModuleList([
          FastSwiGLU(d_model, d_expert, bias=False)
          for _ in range(n_routed_experts)
      ])
      if n_routed_experts == 1:
          self.experts = self.experts[0]

  def forward(self, x):
      if not type(self.shared_experts) == list: shared_experts = [self.shared_experts]
      if not type(self.experts) == list: experts = [self.experts]

      logits = self.router(x) # B, T, n_routed_experts
      probs = F.softmax(logits, dim=-1)

      topk_weights, topk_indices = torch.topk(
          probs,
          self.topk,
          dim=-1
      ) # both are B, T, topk

      # norm
      topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

      shared_out = torch.zeros_like(x) # B, T, d_model
      for exp in shared_experts:
          shared_out += exp(x)

      routed_out = torch.zeros_like(x)
      for i, exp in enumerate(self.experts):
          mask = topk_indices == i # B, T, topk

          if not mask.any(): continue

          # suppose N tokens in total choose this expert

          B, T, topk = mask.nonzero(as_tuple=True) # N
          weight = topk_weights[B, T, topk] # N

          routed_out[B, T] += weight[:, None] * exp(x[B, T]) # N, C

      return shared_out + routed_out # B, T, d_model

In [ ]:
r = torch.randint(0, 1024, (4, 64))

In [ ]:
mymodel = ShibatchV2(ShibatchV2Config)
print(mymodel.num_params)
print(mymodel(r)[0][0])
print(mymodel)

2928392
tensor([[-12.0900, -10.5650,  15.2572,  ..., -15.2338, -10.5377,  -6.9176],
        [  9.1626, -25.3678, -14.9083,  ...,   2.8093, -19.0460, -21.4927],
        [-15.7288,  17.7692,  -5.5754,  ...,  17.6108,   8.7979,   7.2899],
        ...,
        [ -4.3618,  -1.6372,  -3.4536,  ...,  -3.2380,  10.3168,   7.0815],
        [ -6.5164, -10.5611,   3.6211,  ...,  -1.7743, -26.6896,   3.5157],
        [-17.9233,  -3.2889,  12.8575,  ...,  -3.4253,   0.3527,  -5.3947]],
       grad_fn=<SelectBackward0>)
ShibatchV2(
  (model): ModuleDict(
    (embed_tokens): Embedding(1024, 216)
    (layers): ModuleList(
      (0): Block0(
        (post_attention_layernorm): RMSNorm((216,), eps=1e-06, elementwise_affine=True)
        (self_attn): DeepseekMLA(
          (q_proj): Linear(in_features=216, out_features=256, bias=False)
          (kv_a_proj_with_mqa): Linear(in_features=216, out_features=80, bias=False)
          (kv_a_layernorm): RMSNorm((64,), eps=1e-06, elementwise_affine=True)
       

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "shibatch/tinydeepseekv2-3m"
model_dir = Path(snapshot_download(
    repo_id=repo,
    allow_patterns=["hf/*"],
)) / "hf"

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    dtype=torch.float32,
)

print(sum([p.numel() for p in model.parameters()]))
my_logits = mymodel(r)[0]
shibatch_logits = model(r)[0]

diff = shibatch_logits - my_logits

print("max:", diff.abs().max())
print("mean:", diff.abs().mean())
print("rmse:", diff.pow(2).mean().sqrt())
print("cos:", torch.nn.functional.cosine_similarity(
    shibatch_logits.flatten(),
    my_logits.flatten(),
    dim=0
))
# print(model)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/64 [00:00<?, ?it/s]

2928392
max: tensor(36.0653, grad_fn=<MaxBackward1>)
mean: tensor(1.7380, grad_fn=<MeanBackward0>)
rmse: tensor(2.2962, grad_fn=<SqrtBackward0>)
cos: tensor(0.9338, grad_fn=<SumBackward1>)


In [ ]:
def load_weights(to_model: nn.Module, from_model: nn.Module, map: dict | None = None, grouped: dict | None = None, transposed: dict | None = None, filter: list | None = None):
  to_sd = to_model.state_dict()

  from_sd = from_model.state_dict()
  if filter is not None:
    to_sd = {k: v for k, v in to_sd.items() if not any(k.endswith(f) for f in filter)}
    from_sd = {k: v for k, v in from_sd.items() if not any(k.endswith(f) for f in filter)}

  to_sd_keys = to_sd.keys()

  for k in to_sd_keys:
    key = k
    loaded = False

    if grouped is not None:
      for tp, fp_list in grouped.items(): # fp -> from (model) pattern/layername, tp -> to (hf model) pattern/layername
        if k.endswith(f'{tp}.weight' if not (tp.endswith('.bias') or tp.endswith('.weight')) else tp):
          base_key = k.replace(f'{tp}.weight', '')
          from_weights = torch.cat([from_sd[f"{base_key}{i}.weight"] for i in fp_list], 0)

          assert from_weights.shape == to_sd[k].shape, f"Shape does not match for grouped weight {tp} <- {fp_list}"

          with torch.no_grad():
              to_sd[k].copy_(from_weights)

          loaded = True
          break
      if loaded: continue

    if transposed is not None:
      for tp, fp in transposed.items():
        if k.endswith(f'{tp}.weight'):
          key = k.replace(tp, fp)

          if k == 'transformer.h.0.attn.c_attn.bias':
            print('tp')

          assert from_sd[key].t().shape == to_sd[k].shape

          with torch.no_grad():
              to_sd[k].copy_(from_sd[key].t())

          loaded = True
          break
      if loaded: continue

    if map is not None:
      for tp, fp in map.items():
        if k.endswith(f'{tp}.weight' if not (tp.endswith('.bias') or tp.endswith('.weight')) else tp):
          key = k.replace(tp, fp)

          assert from_sd[key].shape == to_sd[k].shape

    assert key in from_sd, f"Missing key: {key}"

    with torch.no_grad():
      to_sd[k].copy_(from_sd[key])

  print('Weights loaded successfully!')

In [ ]:
to_sd = mymodel.state_dict()
from_sd = model.state_dict()

to_sd_keys = to_sd.keys()
from_sd_keys = from_sd.keys()

keys = []
for k in from_sd_keys:
    if '.experts' in k:
        key = '.'.join(k.split('.')[:-1])
        if key not in keys: keys.append(key)

for i, k in enumerate(keys):
    g, d = from_sd[f'{k}.gate_up_proj'][i], from_sd[f'{k}.down_proj'][i]
    keyg, keyd = k + f'.{i}.gatexvalue.weight', k + f'.{i}.out_proj.weight'

    with torch.no_grad():
      to_sd[keyg].copy_(g)
      to_sd[keyd].copy_(d)

filter = []
for i in range(mymodel.config.n_routed_experts):
    filter.extend([f'experts.{i}.gatexvalue.weight', f'experts.{i}.out_proj.weight'])

load_weights(mymodel, model, map={
    'mlp0.out_proj': 'mlp.down_proj',
    'shared_experts.out_proj': 'shared_experts.down_proj',
    'router': 'gate'
},
grouped={
    '0.mlp0.gatexvalue': ['0.mlp.gate_proj', '0.mlp.up_proj'],
    'shared_experts.gatexvalue': ['shared_experts.gate_proj', 'shared_experts.up_proj']
},
filter=filter
)

Weights loaded successfully!
